# Week 2 - Exploratory Financial Analysis

## Title

## Objective
Use the final cleaned retail finance dataset to explore revenue, expense, profit, product, regional, and time-series patterns and identify business-relevant trends.

## Dataset Overview

The dataset used for Week 2 and Week 3 is the validated `financial_retail_clean.csv` output from Week 1. It contains 850 unique retail transactions with revenue, cost, operating expense, profit, product, category, region, city, quantity, and date fields.

## Business Questions

- What is total revenue and how is it trending?
- Which categories and regions generate the most revenue and profit?
- How do expenses and profit vary over time?
- Which products contribute the most to profitability?
- Which relationships are strongest among the financial variables?

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
workspace_roots = [Path.cwd(), *Path.cwd().parents]
data_path = next(
    (
        root / 'data' / 'cleaned' / 'financial_retail_clean.csv'
        for root in workspace_roots
        if (root / 'data' / 'cleaned' / 'financial_retail_clean.csv').is_file()
    ),
    None,
)
if data_path is None:
    raise FileNotFoundError('Could not find data/cleaned/financial_retail_clean.csv.')
df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])
df['Total_Expense'] = df['Cost'] + df['Operating_Expense']
df['Net_Profit'] = df['Revenue'] - df['Cost'] - df['Operating_Expense']
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
print('Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())
print('Missing values:', df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Revenue total:', round(df['Revenue'].sum(), 2))
print('Net profit total:', round(df['Net_Profit'].sum(), 2))

Rows: 850
Columns: 20
Date range: 2025-01-01 to 2025-12-31
Missing values: Transaction_ID       0
Date                 0
Revenue              0
Cost                 0
Operating_Expense    0
Gross_Profit         0
Net_Profit           0
Product              0
Category             0
Customer_ID          0
Customer_Segment     0
Region               0
City                 0
Quantity             0
Order_Status         0
Month                0
Quarter              0
Year                 0
Profit_Status        0
Total_Expense        0
dtype: int64
Duplicate rows: 0
Revenue total: 61039689.51
Net profit total: 13918375.6


## Data Validation

The dataset was checked for rows, columns, missing values, date range, duplicates, and financial consistency before analysis. All validation checks completed successfully with no missing cells and no duplicate rows.

In [2]:
summary = df[['Revenue','Cost','Operating_Expense','Net_Profit','Quantity']].describe().round(2)
print(summary.to_string())
print('Correlation matrix:\n', df[['Revenue','Cost','Operating_Expense','Net_Profit','Quantity']].corr().round(3).to_string())

         Revenue       Cost  Operating_Expense  Net_Profit  Quantity
count     850.00     850.00             850.00      850.00    850.00
mean    71811.40   46398.04            9038.80    16374.56      3.52
std     89823.83   57779.71           12372.76    22397.11      5.52
min       977.22     648.19              93.43       83.03      1.00
25%     15044.63    9566.59            1741.76     2833.05      2.00
50%     40662.92   26603.42            4838.60     8182.99      3.00
75%     91838.49   60663.75           10943.42    20894.06      4.00
max    677464.15  430832.76          113021.81   169757.34    100.00
Correlation matrix:
                    Revenue   Cost  Operating_Expense  Net_Profit  Quantity
Revenue              1.000  0.992              0.960       0.922     0.199
Cost                 0.992  1.000              0.956       0.868     0.195
Operating_Expense    0.960  0.956              1.000       0.830     0.177
Net_Profit           0.922  0.868              0.830      

## Revenue Analysis

This section reviews the total revenue and the revenue mix across months, categories, and regions.

In [3]:
monthly_revenue = df.groupby(df['Date'].dt.strftime('%B'), sort=False)['Revenue'].sum().reset_index(name='Revenue')
print(monthly_revenue.to_string(index=False))
category_revenue = df.groupby('Category')['Revenue'].sum().sort_values(ascending=False)
print('Revenue by category:\n', category_revenue.to_string())
region_revenue = df.groupby('Region')['Revenue'].sum().sort_values(ascending=False)
print('Revenue by region:\n', region_revenue.to_string())

     Date    Revenue
  January 4865781.43
 February 5508151.11
    March 4383197.83
    April 4982268.37
      May 5884249.97
     June 6296528.44
     July 5391616.52
   August 5092172.61
September 4373313.75
  October 5762384.55
 November 4043125.98
 December 4456898.95
Revenue by category:
 Category
Electronics    52680285.38
Furniture       5152662.82
Accessories     3206741.31
Revenue by region:
 Region
West     34309749.58
South    13379888.38
North    10818323.38
East      2531728.17


## Expense Analysis

Monthly and category-level cost patterns help explain the underlying production and operating structure.

In [4]:
monthly_expense = df.groupby(df['Date'].dt.strftime('%B'), sort=False)['Cost'].sum().reset_index(name='Expense')
print(monthly_expense.to_string(index=False))
category_expense = df.groupby('Category')['Cost'].sum().sort_values(ascending=False)
print('Expense by category:\n', category_expense.to_string())
region_expense = df.groupby('Region')['Cost'].sum().sort_values(ascending=False)
print('Expense by region:\n', region_expense.to_string())

     Date    Expense
  January 3166458.79
 February 3551526.94
    March 2886755.42
    April 3215932.19
      May 3864012.01
     June 3997287.39
     July 3448324.56
   August 3342602.62
September 2716050.52
  October 3837206.03
 November 2545324.17
 December 2866854.60
Expense by category:
 Category
Electronics    33938664.76
Furniture       3428248.48
Accessories     2071422.00
Expense by region:
 Region
West     22041240.06
South     8628701.11
North     7091409.58
East      1676984.49


## Profit Analysis

Net profit provides the clearest view of business value after direct cost and operating expense are considered.

In [5]:
category_profit = df.groupby('Category')['Net_Profit'].sum().sort_values(ascending=False)
print('Profit by category:', category_profit.to_string())
region_profit = df.groupby('Region')['Net_Profit'].sum().sort_values(ascending=False)
print('Profit by region:\n', region_profit.to_string())
print('Overall profit margin:', round(df['Net_Profit'].sum()/df['Revenue'].sum()*100,2), '%')

Profit by category: Category
Electronics    12065201.32
Furniture       1110148.69
Accessories      743025.59
Profit by region:
 Region
West     7924155.68
South    3056702.72
North    2370668.54
East      566848.66
Overall profit margin: 22.8 %


## Product and Category Analysis

Product granularity helps identify which items drive revenue and margin in a financial retail portfolio.

In [6]:
product_summary = df.groupby('Product').agg(Revenue=('Revenue','sum'), Profit=('Net_Profit','sum'), Quantity=('Quantity','sum')).sort_values('Revenue', ascending=False)
print(product_summary.head(10).to_string())
product_margin = product_summary.assign(Margin=lambda x: x['Profit'] / x['Revenue'] * 100).sort_values('Margin', ascending=False)
print('Highest-margin products:\n', product_margin.head(10).to_string())

                  Revenue      Profit  Quantity
Product                                        
Laptop        28023050.88  6416577.50       435
Smartphone    13624053.70  3001913.64       664
Tablet         4273728.49  1017224.16       189
Monitor        3947910.29   897454.71       303
Desk           3185392.74   659425.27       181
Printer        2232128.32   537671.44       238
Office Chair   1967270.08   450723.42       162
Smartwatch     1279243.67   313150.06       147
Headphones     1174588.23   252871.28       264
Unknown         603622.26   201570.03        17
Highest-margin products:
                   Revenue      Profit  Quantity     Margin
Product                                                   
Unknown         603622.26   201570.03        17  33.393406
Keyboard        310723.98    77473.28       169  24.933151
Smartwatch     1279243.67   313150.06       147  24.479313
Printer        2232128.32   537671.44       238  24.087837
Tablet         4273728.49  1017224.16       

## Regional Analysis

Regional performance helps identify where the business is strongest and where profitability is weaker.

In [7]:
region_metrics = df.groupby('Region').agg(Revenue=('Revenue','sum'), Expense=('Cost','sum'), Profit=('Net_Profit','sum')).assign(Margin=lambda x: x['Profit']/x['Revenue']*100)
print(region_metrics.sort_values('Revenue', ascending=False).round(2).to_string())

            Revenue      Expense      Profit  Margin
Region                                              
West    34309749.58  22041240.06  7924155.68   23.10
South   13379888.38   8628701.11  3056702.72   22.85
North   10818323.38   7091409.58  2370668.54   21.91
East     2531728.17   1676984.49   566848.66   22.39


## Monthly and Time-Based Analysis

Time-based analysis is essential to find seasonality and the strongest/weakest operating periods.

In [8]:
month_summary = df.groupby(df['Date'].dt.to_period('M').astype(str)).agg(Revenue=('Revenue','sum'), Expense=('Cost','sum'), Profit=('Net_Profit','sum')).reset_index()
month_summary['Revenue_Growth_pct'] = month_summary['Revenue'].pct_change().mul(100)
month_summary['Profit_Growth_pct'] = month_summary['Profit'].pct_change().mul(100)
print(month_summary.round(2).to_string(index=False))

   Date    Revenue    Expense     Profit  Revenue_Growth_pct  Profit_Growth_pct
2025-01 4865781.43 3166458.79 1069866.32                 NaN                NaN
2025-02 5508151.11 3551526.94 1301465.73               13.20              21.65
2025-03 4383197.83 2886755.42  948882.90              -20.42             -27.09
2025-04 4982268.37 3215932.19 1162206.42               13.67              22.48
2025-05 5884249.97 3864012.01 1291035.06               18.10              11.08
2025-06 6296528.44 3997287.39 1429734.27                7.01              10.74
2025-07 5391616.52 3448324.56 1305130.24              -14.37              -8.72
2025-08 5092172.61 3342602.62 1117097.24               -5.55             -14.41
2025-09 4373313.75 2716050.52 1118001.08              -14.12               0.08
2025-10 5762384.55 3837206.03 1166436.00               31.76               4.33
2025-11 4043125.98 2545324.17  965285.36              -29.84             -17.24
2025-12 4456898.95 2866854.60 1043234.98

## Transaction Analysis

Transaction analysis looks at the typical sales size and the distribution of extremes in the dataset.

In [9]:
avg_transaction = df['Revenue'].mean()
print('Average transaction value:', round(avg_transaction, 2))
print('Median transaction value:', round(df['Revenue'].median(), 2))
print('Max transaction value:', round(df['Revenue'].max(), 2))
print('Min transaction value:', round(df['Revenue'].min(), 2))
print('Transaction count:', len(df))

Average transaction value: 71811.4
Median transaction value: 40662.92
Max transaction value: 677464.15
Min transaction value: 977.22
Transaction count: 850


## Distribution Analysis

The distribution of revenue and profit indicates whether results are concentrated or spread out across transactions.

In [10]:
print('Revenue skewness:', round(df['Revenue'].skew(), 3))
print('Profit skewness:', round(df['Net_Profit'].skew(), 3))
print('Revenue variance:', round(df['Revenue'].var(), 2))
print('Profit variance:', round(df['Net_Profit'].var(), 2))

Revenue skewness: 2.689
Profit skewness: 2.942
Revenue variance: 8068320556.89
Profit variance: 501630328.83


## Correlation Analysis

Correlation coefficients show how strongly financial variables move together without implying causation.

In [11]:
corr = df[['Revenue','Cost','Operating_Expense','Net_Profit','Quantity']].corr().round(3)
print(corr.to_string())

                   Revenue   Cost  Operating_Expense  Net_Profit  Quantity
Revenue              1.000  0.992              0.960       0.922     0.199
Cost                 0.992  1.000              0.956       0.868     0.195
Operating_Expense    0.960  0.956              1.000       0.830     0.177
Net_Profit           0.922  0.868              0.830       1.000     0.197
Quantity             0.199  0.195              0.177       0.197     1.000


## Trend Analysis

Trend analysis across months helps identify whether the business is expanding, contracting, or remaining stable.

In [12]:
trend = df.groupby(df['Date'].dt.to_period('M').astype(str)).agg(Revenue=('Revenue','sum'), Profit=('Net_Profit','sum')).reset_index()
trend['Revenue_Growth_pct'] = trend['Revenue'].pct_change().mul(100)
trend['Profit_Growth_pct'] = trend['Profit'].pct_change().mul(100)
print(trend.round(2).to_string(index=False))

   Date    Revenue     Profit  Revenue_Growth_pct  Profit_Growth_pct
2025-01 4865781.43 1069866.32                 NaN                NaN
2025-02 5508151.11 1301465.73               13.20              21.65
2025-03 4383197.83  948882.90              -20.42             -27.09
2025-04 4982268.37 1162206.42               13.67              22.48
2025-05 5884249.97 1291035.06               18.10              11.08
2025-06 6296528.44 1429734.27                7.01              10.74
2025-07 5391616.52 1305130.24              -14.37              -8.72
2025-08 5092172.61 1117097.24               -5.55             -14.41
2025-09 4373313.75 1118001.08              -14.12               0.08
2025-10 5762384.55 1166436.00               31.76               4.33
2025-11 4043125.98  965285.36              -29.84             -17.24
2025-12 4456898.95 1043234.98               10.23               8.08


## Key Findings

1. Electronics is the largest revenue and profit category.
2. West is the highest-performing region.
3. Accessories have the strongest margin among categories.
4. Laptop is the core product line.
5. Revenue peaks in late spring and early summer.
6. Revenue and cost are highly correlated.
7. Average transaction size is materially above the minimum.
8. The dataset contains no loss-making transactions in the final cleaned set.

## Business Interpretation

The pattern indicates a retail model whose overall performance is led by electronics and the West region. The strongest operational opportunities lie in preserving margin quality, protecting category balance, and planning for seasonal variation.

## Conclusion

The cleaned financial-retail dataset supports a detailed EDA and profitability review. The most important commercial findings are the dominance of Electronics, the leadership of the West region, the margin strength of Accessories, and the seasonal swings in revenue and profit performance.